# ModernBERT Decoder: teach a 17M pilot

**Mission:** turn a text description of Cinder Station into one of five actions:
`attack`, `heal`, `resupply`, `collect`, `extract`.
We train the backbone and decision head using Laya's noisy-logit proper-scoring
RLCD objective **plus cross-entropy**. The game motor handles navigation and aiming.

Open **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
Use a fresh runtime. No API key or Google Drive mount is required. This downloads
the public **jhu-clsp/ettin-decoder-17m** checkpoint and installs pinned Python packages.

Reference T4 run: **40/200 → 183/200 (20% → 91.5%)**, **45.38 seconds** for training/validation/checkpoint work,
excluding installation and initial model download/load. Peak CUDA memory: **0.369 GiB allocated / 0.408 GiB reserved**.
These are archived observations, not promised results for this run.
Colab availability, runtime packages and hardware can change.

[Project and Doom-inspired game](https://github.com/shyamsridhar123/LAYA-RLCD) ·
[Evidence and limitations](https://github.com/shyamsridhar123/LAYA-RLCD/blob/main/docs/BENCHMARKS.md) ·
[Learning guide](https://github.com/shyamsridhar123/LAYA-RLCD/blob/main/docs/LEARNING.md)

This is a new public wrapper around the retained training source. The reference
training run is documented separately; notebook outputs start empty.

## 1. Install the experiment recipe

Keep Colab's CUDA-enabled PyTorch. Pin the experiment packages and constrain pip
to the already-installed Torch version, so dependency resolution cannot silently
replace it. The reference environment used Python 3.13.15, Torch 2.11.0+cu128,
CUDA 12.8 and a Tesla T4. A different runtime is a replication, not a bitwise replay.
If an import fails after changing packages, restart the runtime and run from here.

In [ ]:
import os, sys, subprocess, importlib.metadata
from pathlib import Path
assert (3, 11) <= sys.version_info < (3, 14), "Use Python 3.11–3.13 for this recipe."
os.environ.update(USE_TF="0", USE_FLAX="0", TOKENIZERS_PARALLELISM="false",
                  HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_PROGRESS_BARS="1")
torch_version = importlib.metadata.version("torch")
constraint = Path("/tmp/laya-rlcd-torch-constraint.txt")
constraint.write_text(f"torch=={torch_version}\n")
packages = ["laya==0.3.4", "transformers==5.0.0", "huggingface_hub==1.29.0",
            "safetensors==0.8.0", "numpy==2.1.3"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraint), *packages], check=True)
import torch
assert torch.cuda.is_available(), "Select a GPU runtime, then start again."
print({"torch": torch.__version__, "cuda": torch.version.cuda,
       "gpu": torch.cuda.get_device_name(0)})

## 2. Unpack the public, hashed lesson files

The next cell embeds only the readable training script and six JSON protocol/data
files from this repository. The builder is `scripts/build_notebooks.py`; it uses
an explicit file list. Each run gets a new temporary directory.
No personal Drive files, notebook history, keys, or model weights are included.

In [ ]:
import base64, hashlib, io, json, tempfile, zipfile
from pathlib import Path

PAYLOAD = '...'
MANIFEST = {'data/gate.json': '5f124a65f0d79772549684137b5c5d0a214dbcccd55e4a6b8606c3f6647c2f1f', 'data/protocol.json': '3ea89efd6c776f949352fadda1a65749c664fb86dbcb4635e8ec91354fe6fac2', 'data/temperature.json': '382546c43e22047bd78200201e602472fa6294d0a60c8c3141aaf41204e55cd8', 'data/test.json': '7057f49b6dc47cd3e79fda2f778d85e3ccf794f114d6cf76c40fcae831447bb8', 'data/train.json': '7060a50604ad3fac3c391e45ff398dba535882065ad4e184e081f06122e2d89b', 'data/validation.json': 'c1d8833147b91fd2c4f87ce0545675524885218e8b0d5d3ca02f4a5622610c93', 'scripts/modernbert_decoder_colab.py': '5713f0aea541685e9beb6ceae92b25517b03227b9a0415e3d00dd81e3127cb1f'}
ROOT = Path(tempfile.mkdtemp(prefix="laya-rlcd-decoder-"))
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD))) as archive:
    assert set(archive.namelist()) == set(MANIFEST)
    for name, digest in MANIFEST.items():
        content = archive.read(name)
        assert hashlib.sha256(content).hexdigest() == digest, name
        target = (ROOT / name).resolve()
        assert target.is_relative_to(ROOT.resolve())
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)
print(f"Verified {len(MANIFEST)} public files in a fresh experiment directory.")

## 3. Inspect the rules before training

Rule priority is the label source: heal below 35 HP with a medkit; otherwise
resupply at 2 or fewer rounds when hostiles and crates remain; otherwise attack;
then collect the core and extract. Read one **training** example.

Splits are balanced and disjoint by the six decision-relevant state variables:
500 train, 100 validation, 100 temperature, 100 gate, 200 test.
The decoder uses only train/validation/test. Laya calibrates on the two separate
100-example splits after selecting its checkpoint. Never select on test accuracy.

In [ ]:
protocol = json.loads((ROOT / "data/protocol.json").read_text())
training_rows = json.loads((ROOT / "data/train.json").read_text())
print(protocol["question"]["instructions"])
print({name: spec["n"] for name, spec in protocol["splits"].items()})
print(json.dumps(training_rows[0], indent=2))
print("Training configuration:", protocol["training"])

## 4. Train the pilot

Four epochs, microbatch 2, accumulation 4, backbone learning rate 2.5e-5,
head learning rate 1e-4, four reward samples, sigma 0.4 → 0.1.
Checkpoint selection maximizes validation accuracy, breaking ties with NLL.

The decoder uses FP32 because a preliminary FP16 attempt overflowed gradients.
Laya keeps FP32 trainable weights with FP16 autocast, then saves FP16 weights.
Expect 252 optimizer updates. If a run skips updates, keep that fact in its report.
This is supervised state-label learning with RLCD+CE, not reward from playing missions.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts/modernbert_decoder_colab.py"),
                "train", "--root", str(ROOT)], check=True)
training = json.loads((ROOT / "results/training.json").read_text())
print("Selected epoch:", training["selected_epoch"])
print("Validation:", training["selected_validation"])

## 5. Freeze selection, then evaluate

Only now open the 200-example test set. Save every prediction, label, probability
and latency. Accuracy asks whether the action matches the rule; latency includes
tokenization and inference at batch one after warmup, with CUDA synchronization.
It excludes the browser, local HTTP service, initial load, and installation.

The initial checkpoint has a new random classification head. It is not a zero-shot text-generation baseline. Probabilities are uncalibrated.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "scripts/modernbert_decoder_colab.py"),
                "evaluate", "--root", str(ROOT)], check=True)
result = json.loads((ROOT / "results/evaluation.json").read_text())
print(json.dumps(result, indent=2))

## 6. Take your pilot home

Download **both** the results and selected checkpoint before ending the Colab runtime.
The checkpoint ZIP has a `selected/` folder. Unpack it under `models/` in your local
checkout, then start the game using the command in the next cell.
Results include synthetic states and environment versions. Review any modifications
before sharing; keep credentials and personal data out of your notebook.

In [ ]:
import zipfile
from google.colab import files
exports = {
    "decoder-results.zip": ["results", "data", "scripts"],
    "decoder-checkpoint.zip": ["selected"],
}
for name, folders in exports.items():
    archive_path = ROOT / name
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for folder in folders:
            for path in sorted((ROOT / folder).rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(ROOT).as_posix())
    print(name, round(archive_path.stat().st_size / 2**20, 1), "MiB")
    files.download(str(archive_path))
print("On your computer: python scripts/setup.py --models")
print("Then: python scripts/play.py --model decoder --model-dir models/selected")

## Bonus missions

- **Find the boundary bug.** Compare HP 34 vs 35 and ammo 2 vs 3. Change one fact at a time.
- **Earn the RLCD claim.** Compare CE-only to RLCD+CE with multiple seeds and the same data budget.
  The reference runs do not isolate RLCD's contribution.
- **Break the template.** Rewrite the state text while preserving meaning, then test new maps.
- **Calibrate your confidence.** Compare top probability, normalized entropy and actual error rate.

Keep the published reference data unchanged. Fork the experiment, define a new test set
before looking at its outcomes, and publish errors alongside successes.